In [ ]:
!nvidia-smi

To enable this option, you must change **Go to Runtime -> Change runtime type , and select “GPU” as the Hardware Accelerator**

## Install Ultralytics

In [ ]:
%pip install "ultralytics==8.4.215" "PyYAML>=6"

## Prepare the dataset

By default, this cell downloads and extracts the dataset from GitHub into the Colab runtime. To use a copy in My Drive, uncomment the three Google Drive lines.

In [ ]:
from pathlib import Path
from tempfile import TemporaryDirectory
from urllib.request import urlretrieve
from zipfile import ZipFile

import yaml
from ultralytics import YOLO

local_root = Path("/content/D-Fire-datasets")
dataset = local_root / "fire"

# Google Drive option
# from google.colab import drive
# drive.mount("/content/drive")
# dataset = Path("/content/drive/MyDrive/D-Fire-datasets/fire")

if dataset == local_root / "fire" and not local_root.exists():
    dataset_url = "https://github.com/ZhengLinLei/FireGuardML/releases/download/v1/D-Fire-datasets.zip"
    with TemporaryDirectory(dir=local_root.parent) as temp_dir:
        archive = Path(temp_dir) / "D-Fire-datasets.zip"
        urlretrieve(dataset_url, archive)
        with ZipFile(archive) as zip_file:
            zip_file.extractall(temp_dir)
        (Path(temp_dir) / local_root.name).rename(local_root)

# The YAML in the ZIP uses absolute Google Drive paths.
config = yaml.safe_load((dataset / "data.yaml").read_text(encoding="utf-8"))
config.update(path=str(dataset), train="train/images", val="valid/images", test="test/images")
dataset_yaml = dataset / "fireguard-colab.yaml"
dataset_yaml.write_text(yaml.safe_dump(config, sort_keys=False), encoding="utf-8")

## Train YOLO26s

In [ ]:
model = YOLO("yolo26s.pt")
model.train(
    data=str(dataset_yaml),
    epochs=25,
    imgsz=416,
    plots=True,
    project=str(dataset / "runs" / "detect"),
    name="fireguard-yolo26s",
)
run_dir = model.trainer.save_dir
best_weights = model.trainer.best
print(f"Best checkpoint: {best_weights}")

## View training results

In [ ]:
from IPython.display import Image, display

for filename in ("confusion_matrix.png", "results.png", "val_batch0_labels.jpg"):
    plot = run_dir / filename
    if plot.is_file():
        display(Image(filename=str(plot), width=600))

## Run inference with the trained checkpoint

In [ ]:
image = next((dataset / "valid" / "images").glob("*.jpg"))
result = YOLO(best_weights).predict(source=str(image), imgsz=416)[0]
display(result.plot(pil=True))

## Download the updated `doc/` files

Run this cell after training to download a ZIP with the metrics, plots, and example images from the current run. Extract it at the repository root to update `doc/`.

In [ ]:
from zipfile import ZIP_DEFLATED, ZipFile

import matplotlib.pyplot as plt
import numpy as np
from google.colab import files

# Rebuild the bounding-box correlogram from this run's training labels.
labels = model.trainer.train_loader.dataset.labels
boxes = np.concatenate([label["bboxes"] for label in labels])
fields = ("x center", "y center", "width", "height")
fig, axes = plt.subplots(4, 4, figsize=(10, 10), layout="constrained")
for row, y_name in enumerate(fields):
    for col, x_name in enumerate(fields):
        ax = axes[row, col]
        if row == col:
            ax.hist(boxes[:, col], bins=30, range=(0, 1))
        else:
            ax.hist2d(boxes[:, col], boxes[:, row], bins=30, range=((0, 1), (0, 1)), cmap="Blues")
        if row == len(fields) - 1:
            ax.set_xlabel(x_name)
        if col == 0:
            ax.set_ylabel(y_name)
fig.savefig(run_dir / "labels_correlogram.jpg", dpi=200)
plt.close(fig)

# Keep the filenames used by the repository, even if batch numbers change.
train_preview = max(
    run_dir.glob("train_batch[0-9]*.jpg"),
    key=lambda path: int(path.stem.removeprefix("train_batch")),
)
doc_files = {
    "results.csv": "results.csv",
    "results.png": "results.png",
    "confusion_matrix_normalized.png": "confusion_matrix_normalized.png",
    "PR_curve.png": "BoxPR_curve.png",
    "P_curve.png": "BoxP_curve.png",
    "R_curve.png": "BoxR_curve.png",
    "labels.jpg": "labels.jpg",
    "labels_correlogram.jpg": "labels_correlogram.jpg",
    "train_batch827.jpg": train_preview.name,
    "val_batch0_pred.jpg": "val_batch0_pred.jpg",
}

archive_path = run_dir / "FireGuardML-YOLO26s-doc.zip"
with ZipFile(archive_path, "w", compression=ZIP_DEFLATED) as archive:
    for doc_name, run_name in doc_files.items():
        archive.write(run_dir / run_name, f"doc/{doc_name}")

files.download(str(archive_path))